In [ ]:
import os
from typing import TypedDict, Annotated, Optional, Literal
from operator import add
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
class AdaptiveState(TypedDict):
    query: str
    route: str
    documents: Annotated[list[str], add]
    web_results: Annotated[list[str], add]
    sub_queries: Annotated[list[str], add]
    retrieved: Annotated[list[str], add]
    hop_count: int
    max_hops: int
    answer: str

def init_state(q: str, max_hops: int = 3) -> dict:
    return {"query": q, "route": "", "documents": [], "web_results": [],
            "sub_queries": [], "retrieved": [], "hop_count": 0,
            "max_hops": max_hops, "answer": ""}


In [ ]:
class _Route(BaseModel):
    """쿼리를 처리 경로로 분류."""
    route: Literal["direct", "rag", "web", "multihop"] = Field(
        description="direct=상식, rag=사내문서, web=최신정보, multihop=복합추론"
    )

router_llm = llm.with_structured_output(_Route)

def classify_node(state: AdaptiveState) -> dict:
    sys = ("질문을 분류하세요. direct=LLM 상식, rag=사내 문서, "
           "web=최신/실시간, multihop=여러 단계 추론이 필요한 복합 질문.")
    r = router_llm.invoke([SystemMessage(content=sys),
                            HumanMessage(content=state["query"])])
    return {"route": r.route}

In [ ]:
builder = StateGraph(AdaptiveState)
builder.add_node("classify", classify_node)
builder.add_edge(START, "classify")
builder.add_edge("classify", END)
app_v2 = builder.compile()

queries = [
    "파이썬 리스트가 뭐야?",
    "우리 회사 휴가 정책은?",
    "오늘 코스피 지수는?",
    "봉준호 2019년 영화의 음악감독 출생연도는?",
]
for q in queries:
    r = app_v2.invoke(init_state(q))
    print(f"{q[:25]:<25} → route={r['route']}")

In [ ]:
def direct_node(state: AdaptiveState) -> dict:
    r = llm.invoke([
        SystemMessage(content="간결히 답변하세요."),
        HumanMessage(content=state["query"]),
    ])
    return {"answer": r.content}

def route_fn(state: AdaptiveState) -> str:
    return state["route"]

In [ ]:
builder = StateGraph(AdaptiveState)
builder.add_node("classify", classify_node)
builder.add_node("direct", direct_node)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route_fn, {
    "direct": "direct",
    "rag": END,       # 아직 미구현
    "web": END,
    "multihop": END,
})
builder.add_edge("direct", END)
app_v3 = builder.compile()

# direct로 분류될 쿼리만 진짜로 답함
r = app_v3.invoke(init_state("파이썬 리스트가 뭐야?"))
print(f"route={r['route']}\nanswer={r['answer'][:80]}")

In [ ]:
DOCS = [
    "휴가 정책: 입사 1년 미만은 월 1일, 1년 이상은 연 15일 유급 휴가가 부여됩니다.",
    "출장 규정: 국내 출장은 일비 5만원, 해외는 일비 10만원이 지급됩니다.",
    "재택근무: 주 2일까지 재택근무 가능. 팀장 사전 승인 필요.",
    "야근 식대: 오후 8시 이후 야근 시 1만원 식대 지급.",
    "교육비 지원: 직무 관련 도서/강의 연 30만원 한도 지원.",
    "건강검진: 매년 1회 종합검진 무료 제공.",
    "복장 규정: 비즈니스 캐주얼. 고객 미팅 시 정장 권장.",
]
